#Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import date

#Get Data From Bronze

In [0]:
# Load sales data from Azure Data Lake Storage
df = spark.read.format("parquet")\
        .option("inferSchema",True)\
        .load("abfss://bronze@storenapatdatalake.dfs.core.windows.net/crm_sales_details")
df.limit(10).display()

sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
SO43697,BK-R93R-62,21768,20101229,20110105,20110110,3578,1,3578
SO43698,BK-M82S-44,28389,20101229,20110105,20110110,3400,1,3400
SO43699,BK-M82S-44,25863,20101229,20110105,20110110,3400,1,3400
SO43700,BK-R50B-62,14501,20101229,20110105,20110110,699,1,699
SO43701,BK-M82S-44,11003,20101229,20110105,20110110,3400,1,3400
SO43702,BK-R93R-44,27645,20101230,20110106,20110111,3578,1,3578
SO43703,BK-R93R-62,16624,20101230,20110106,20110111,3578,1,3578
SO43704,BK-M82B-48,11005,20101230,20110106,20110111,3375,1,3375
SO43705,BK-M82S-38,11011,20101230,20110106,20110111,3400,1,3400
SO43706,BK-R93R-48,27621,20101231,20110107,20110112,3578,1,3578


#Data Transformation

##Trimming

In [0]:
# Remove leading and trailing spaces from string columns
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,trim(col(field.name)))

In [0]:
df.limit(10).display()

sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
SO43697,BK-R93R-62,21768,20101229,20110105,20110110,3578,1,3578
SO43698,BK-M82S-44,28389,20101229,20110105,20110110,3400,1,3400
SO43699,BK-M82S-44,25863,20101229,20110105,20110110,3400,1,3400
SO43700,BK-R50B-62,14501,20101229,20110105,20110110,699,1,699
SO43701,BK-M82S-44,11003,20101229,20110105,20110110,3400,1,3400
SO43702,BK-R93R-44,27645,20101230,20110106,20110111,3578,1,3578
SO43703,BK-R93R-62,16624,20101230,20110106,20110111,3578,1,3578
SO43704,BK-M82B-48,11005,20101230,20110106,20110111,3375,1,3375
SO43705,BK-M82S-38,11011,20101230,20110106,20110111,3400,1,3400
SO43706,BK-R93R-48,27621,20101231,20110107,20110112,3578,1,3578


##Cleaning Dates

In [0]:
# Convert valid YYYYMMDD values to dates and invalid values to null
df = df.withColumn(
        "sls_order_dt", when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8), None
        ).otherwise(to_date(col("sls_order_dt").cast("string"),"yyyyMMdd"))
    ).withColumn(
        "sls_ship_dt", when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8), None
        ).otherwise(to_date(col("sls_ship_dt").cast("string"),"yyyyMMdd"))
    ).withColumn(
        "sls_due_dt", when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8), None
        ).otherwise(to_date(col("sls_due_dt").cast("string"),"yyyyMMdd"))
    )

In [0]:
df.limit(10).display()

sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
SO43697,BK-R93R-62,21768,2010-12-29,2011-01-05,2011-01-10,3578,1,3578
SO43698,BK-M82S-44,28389,2010-12-29,2011-01-05,2011-01-10,3400,1,3400
SO43699,BK-M82S-44,25863,2010-12-29,2011-01-05,2011-01-10,3400,1,3400
SO43700,BK-R50B-62,14501,2010-12-29,2011-01-05,2011-01-10,699,1,699
SO43701,BK-M82S-44,11003,2010-12-29,2011-01-05,2011-01-10,3400,1,3400
SO43702,BK-R93R-44,27645,2010-12-30,2011-01-06,2011-01-11,3578,1,3578
SO43703,BK-R93R-62,16624,2010-12-30,2011-01-06,2011-01-11,3578,1,3578
SO43704,BK-M82B-48,11005,2010-12-30,2011-01-06,2011-01-11,3375,1,3375
SO43705,BK-M82S-38,11011,2010-12-30,2011-01-06,2011-01-11,3400,1,3400
SO43706,BK-R93R-48,27621,2010-12-31,2011-01-07,2011-01-12,3578,1,3578


##Sales and Price Corrections

In [0]:
# Correct invalid prices and recalculate inconsistent sales amounts
df = df.withColumn(
        "sls_price", when(
            (col("sls_price") <= 0) | (col("sls_price").isNull()),
            when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)
        ).otherwise(col("sls_price"))
    ).withColumn(
        "sls_sales",when(
            (col("sls_sales") <= 0) | (col("sls_sales").isNull() | (col("sls_sales") != (abs(col("sls_price")) * col("sls_quantity")))),
                col("sls_price") * col("sls_quantity")
        ).otherwise(col("sls_sales"))
    )

In [0]:
df.limit(10).display()

sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
SO43697,BK-R93R-62,21768,2010-12-29,2011-01-05,2011-01-10,3578.0,1,3578.0
SO43698,BK-M82S-44,28389,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0
SO43699,BK-M82S-44,25863,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0
SO43700,BK-R50B-62,14501,2010-12-29,2011-01-05,2011-01-10,699.0,1,699.0
SO43701,BK-M82S-44,11003,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0
SO43702,BK-R93R-44,27645,2010-12-30,2011-01-06,2011-01-11,3578.0,1,3578.0
SO43703,BK-R93R-62,16624,2010-12-30,2011-01-06,2011-01-11,3578.0,1,3578.0
SO43704,BK-M82B-48,11005,2010-12-30,2011-01-06,2011-01-11,3375.0,1,3375.0
SO43705,BK-M82S-38,11011,2010-12-30,2011-01-06,2011-01-11,3400.0,1,3400.0
SO43706,BK-R93R-48,27621,2010-12-31,2011-01-07,2011-01-12,3578.0,1,3578.0


##Renaming Columns

In [0]:
# Rename source fields to business-friendly names
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old_name , new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)
df.limit(10).display()

order_number,product_number,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price
SO43697,BK-R93R-62,21768,2010-12-29,2011-01-05,2011-01-10,3578.0,1,3578.0
SO43698,BK-M82S-44,28389,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0
SO43699,BK-M82S-44,25863,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0
SO43700,BK-R50B-62,14501,2010-12-29,2011-01-05,2011-01-10,699.0,1,699.0
SO43701,BK-M82S-44,11003,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0
SO43702,BK-R93R-44,27645,2010-12-30,2011-01-06,2011-01-11,3578.0,1,3578.0
SO43703,BK-R93R-62,16624,2010-12-30,2011-01-06,2011-01-11,3578.0,1,3578.0
SO43704,BK-M82B-48,11005,2010-12-30,2011-01-06,2011-01-11,3375.0,1,3375.0
SO43705,BK-M82S-38,11011,2010-12-30,2011-01-06,2011-01-11,3400.0,1,3400.0
SO43706,BK-R93R-48,27621,2010-12-31,2011-01-07,2011-01-12,3578.0,1,3578.0


##Create silver record_priority to filter out duplicate id

In [0]:
# Rank records using all sales fields to identify duplicates
w = Window.partitionBy(df["order_number"],df["product_number"],df["customer_id"],df["order_date"],
                       df["ship_date"],df["due_date"],df["sales_amount"],df["quantity"],df["price"]).orderBy(df["order_date"].desc_nulls_last())
df = df.withColumn("record_priority",row_number().over(w))

In [0]:
df.limit(5).display()

order_number,product_number,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price,record_priority
SO43697,BK-R93R-62,21768,2010-12-29,2011-01-05,2011-01-10,3578.0,1,3578.0,1
SO43698,BK-M82S-44,28389,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0,1
SO43699,BK-M82S-44,25863,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0,1
SO43700,BK-R50B-62,14501,2010-12-29,2011-01-05,2011-01-10,699.0,1,699.0,1
SO43701,BK-M82S-44,11003,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0,1


In [0]:
# Keep one record from each duplicate group
df = df.filter(col("record_priority") == 1)

In [0]:
# Remove any remaining exact duplicate rows
df = df.dropDuplicates()

#Data Writing

In [0]:
# Save cleaned sales data in Parquet format
df.write.format("parquet")\
    .mode("overwrite")\
    .option("path","abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_sales_details")\
    .save()

In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_sales_details` LIMIT 10

order_number,product_number,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price,record_priority
SO43718,BK-R93R-44,27668,2011-01-03,2011-01-10,2011-01-15,3578.0,1,3578.0,1
SO43745,BK-R93R-44,16514,2011-01-10,2011-01-17,2011-01-22,3578.0,1,3578.0,1
SO44431,BK-R93R-48,17725,2011-04-19,2011-04-26,2011-05-01,3578.0,1,3578.0,1
SO45482,BK-R93R-62,12522,2011-08-24,2011-08-31,2011-09-05,3578.0,1,3578.0,1
SO45769,BK-R93R-56,11327,2011-09-26,2011-10-03,2011-10-08,3578.0,1,3578.0,1
SO45903,BK-R93R-62,15619,2011-10-13,2011-10-20,2011-10-25,3578.0,1,3578.0,1
SO46588,BK-M82B-48,20035,2011-12-26,2012-01-02,2012-01-07,3375.0,1,3375.0,1
SO46736,BK-M68B-42,12341,2012-01-04,2012-01-11,2012-01-16,2049.0,1,2049.0,1
SO46764,BK-M68B-42,26238,2012-01-08,2012-01-15,2012-01-20,2049.0,1,2049.0,1
SO47204,BK-R89B-52,25371,2012-02-14,2012-02-21,2012-02-26,2182.0,1,2182.0,1


In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_sales_details` WHERE record_priority > 1 ORDER BY record_priority DESC LIMIT 10

order_number,product_number,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price,record_priority
